In [ ]:
%pip install pandas openpyxl

import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import sqlite3
import pandas as pd
import os

# Database file location
DB_FILE = "mtg_collection.db"

: 

In [ ]:
class MTGCardManager:
    def __init__(self, root):
        self.root = root
        self.root.title("MTG Card Sorter & Inventory")
        self.root.geometry("1300x850")

        self.conn = sqlite3.connect(DB_FILE)
        self.create_table()

        # --- Data Variables ---
        self.var_name = tk.StringVar()
        self.var_color = tk.StringVar()
        self.var_quantity = tk.IntVar(value=1)
        self.var_type = tk.StringVar()
        self.var_mana_cost = tk.StringVar()
        self.var_pt = tk.StringVar()
        self.var_subtypes = tk.StringVar()
        self.var_keywords = tk.StringVar()
        self.var_ability = tk.StringVar()
        self.var_loyalty = tk.StringVar()

        # State variable for Editing
        self.editing_card_id = None 

        # --- Filter Variables ---
        self.search_color = tk.StringVar()
        self.search_cost = tk.StringVar()
        self.search_category = tk.StringVar()
        self.search_subtype = tk.StringVar()
        self.search_keywords = tk.StringVar()

        self.card_types = [
            "Land", "Legendary Creature", "Creature", "Instant", "Sorcery",
            "Legendary Enchantment", "Enchantment", "Planeswalker",
            "Legendary Artifact", "Artifact", "Artifact Equipment",
            "Artifact Vehicle", "Artifact Creature", "Artifact Treasure"
        ]
        
        self.colors = ["White", "Blue", "Black", "Red", "Green", "Colorless", "Multicolor"]

        # Storage for all the treeview widgets
        self.tree_dict = {} 

        self.setup_ui()

    def create_table(self):
        cursor = self.conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS cards (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                color TEXT,
                quantity INTEGER,
                card_category TEXT,
                mana_cost TEXT,
                power_toughness TEXT,
                subtypes TEXT,
                keywords TEXT,
                ability TEXT,
                loyalty INTEGER,
                UNIQUE(name, card_category)
            )
        """)
        self.conn.commit()

    def setup_ui(self):
        # 1. INPUT FRAME
        self.create_input_frame()

        # 2. NOTEBOOK SYSTEM (Tabs)
        self.main_notebook = ttk.Notebook(self.root)
        self.main_notebook.pack(fill="both", expand=True, padx=5, pady=5)

        # -- A. Master Tab --
        master_frame = tk.Frame(self.main_notebook)
        self.main_notebook.add(master_frame, text="  MASTER LIST  ")
        self.create_master_tab_content(master_frame)

        # -- B. Color Tabs --
        for color in self.colors:
            color_frame = tk.Frame(self.main_notebook)
            self.main_notebook.add(color_frame, text=f"  {color}  ")
            self.create_color_tab_content(color_frame, color)

        self.load_all_cards()

    def create_input_frame(self):
        input_frame = tk.LabelFrame(self.root, text="Card Details", padx=10, pady=5)
        input_frame.pack(fill="x", padx=10, pady=5)

        # Row 1: Basic Info
        tk.Label(input_frame, text="Name:").grid(row=0, column=0, sticky="e")
        tk.Entry(input_frame, textvariable=self.var_name, width=25).grid(row=0, column=1, sticky="w")
        
        tk.Label(input_frame, text="Color:").grid(row=0, column=2, sticky="e")
        tk.Entry(input_frame, textvariable=self.var_color, width=15).grid(row=0, column=3, sticky="w")
        
        tk.Label(input_frame, text="Quantity:").grid(row=0, column=4, sticky="e")
        tk.Entry(input_frame, textvariable=self.var_quantity, width=5).grid(row=0, column=5, sticky="w")

        tk.Label(input_frame, text="Category:").grid(row=0, column=6, sticky="e")
        self.type_combo = ttk.Combobox(input_frame, textvariable=self.var_type, values=self.card_types, state="readonly", width=20)
        self.type_combo.grid(row=0, column=7, sticky="w")
        self.type_combo.bind("<<ComboboxSelected>>", self.update_fields_visibility)

        # Row 2: Dynamic Fields
        self.dynamic_frame = tk.Frame(input_frame)
        self.dynamic_frame.grid(row=1, column=0, columnspan=8, pady=5, sticky="w")
        
        self.lbl_mana = tk.Label(self.dynamic_frame, text="Mana Cost:")
        self.ent_mana = tk.Entry(self.dynamic_frame, textvariable=self.var_mana_cost, width=10)
        self.lbl_pt = tk.Label(self.dynamic_frame, text="P/T:")
        self.ent_pt = tk.Entry(self.dynamic_frame, textvariable=self.var_pt, width=8)
        self.lbl_sub = tk.Label(self.dynamic_frame, text="Subtypes:")
        self.ent_sub = tk.Entry(self.dynamic_frame, textvariable=self.var_subtypes, width=15)
        self.lbl_key = tk.Label(self.dynamic_frame, text="Keywords:")
        self.ent_key = tk.Entry(self.dynamic_frame, textvariable=self.var_keywords, width=15)
        self.lbl_ab = tk.Label(self.dynamic_frame, text="Ability:")
        self.ent_ab = tk.Entry(self.dynamic_frame, textvariable=self.var_ability, width=30)
        self.lbl_loy = tk.Label(self.dynamic_frame, text="Loyalty:")
        self.ent_loy = tk.Entry(self.dynamic_frame, textvariable=self.var_loyalty, width=5)

        # Row 3: Buttons
        btn_frame = tk.Frame(input_frame)
        btn_frame.grid(row=2, column=0, columnspan=8, pady=5)
        
        # We keep a reference to the submit button to change its text later
        self.btn_submit = tk.Button(btn_frame, text="Add Card", command=self.add_card, bg="#dddddd", width=15)
        self.btn_submit.pack(side="left", padx=5)

        # DELETE BUTTON
        self.btn_delete = tk.Button(btn_frame, text="Delete Card", command=self.delete_card, bg="#ff9999", width=15)
        self.btn_delete.pack(side="left", padx=5)
        
        tk.Button(btn_frame, text="Clear Form / Cancel", command=self.clear_form, bg="#dddddd", width=15).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Import Excel", command=self.import_excel_dialog, bg="#b3e6b3", width=15).pack(side="left", padx=20)

    def add_card(self):
        """Adds a new card or updates quantity if it exists."""
        name = self.var_name.get().strip()
        cat = self.var_type.get()
        if not name or not cat:
            messagebox.showerror("Error", "Name and Category are required.")
            return

        data = {
            "name": name,
            "color": self.var_color.get(),
            "quantity": self.var_quantity.get(),
            "card_category": cat,
            "mana_cost": self.var_mana_cost.get(),
            "power_toughness": self.var_pt.get(),
            "subtypes": self.var_subtypes.get(),
            "keywords": self.var_keywords.get(),
            "ability": self.var_ability.get(),
            "loyalty": self.var_loyalty.get() if self.var_loyalty.get() else None
        }
        
        action = self.upsert_card(data)
        self.conn.commit()
        if action == "updated": messagebox.showinfo("Success", f"Updated quantity for {name}.")
        else: messagebox.showinfo("Success", f"Added {name}.")
        
        self.load_all_cards()
        self.clear_form()

    def upsert_card(self, data):
        cursor = self.conn.cursor()
        cursor.execute("SELECT id, quantity FROM cards WHERE name=? AND card_category=?", 
                       (data['name'], data['card_category']))
        result = cursor.fetchone()

        if result:
            new_qty = result[1] + data['quantity']
            cursor.execute("UPDATE cards SET quantity=? WHERE id=?", (new_qty, result[0]))
            return "updated"
        else:
            cursor.execute("""
                INSERT INTO cards (name, color, quantity, card_category, mana_cost, power_toughness, subtypes, keywords, ability, loyalty)
                VALUES (:name, :color, :quantity, :card_category, :mana_cost, :power_toughness, :subtypes, :keywords, :ability, :loyalty)
            """, data)
            return "inserted"

    def delete_card(self):
        """Deletes the card currently being edited."""
        if self.editing_card_id is None:
            messagebox.showwarning("Selection Required", "Please double-click a card from the list to select it for deletion.")
            return

        name = self.var_name.get()
        confirm = messagebox.askyesno("Confirm Delete", f"Are you sure you want to permanently delete '{name}'?")
        if confirm:
            try:
                cursor = self.conn.cursor()
                cursor.execute("DELETE FROM cards WHERE id=?", (self.editing_card_id,))
                self.conn.commit()
                messagebox.showinfo("Deleted", "Card removed from collection.")
                self.load_all_cards()
                self.clear_form()
            except Exception as e:
                messagebox.showerror("Error", str(e))

    def create_treeview(self, parent_frame):
        columns = ("Qty", "Name", "Color", "Category", "Cost", "P/T", "Subtypes", "Keywords", "Ability", "Loyalty")
        tree = ttk.Treeview(parent_frame, columns=columns, show="headings")
        
        col_widths = [40, 150, 80, 120, 60, 50, 100, 100, 150, 50]
        for col, width in zip(columns, col_widths):
            tree.heading(col, text=col)
            tree.column(col, width=width)
        
        scrollbar = ttk.Scrollbar(parent_frame, orient=tk.VERTICAL, command=tree.yview)
        tree.configure(yscroll=scrollbar.set)
        scrollbar.pack(side="right", fill="y")
        tree.pack(side="left", fill="both", expand=True)
        
        # BIND DOUBLE CLICK EVENT
        tree.bind("<Double-1>", self.on_tree_double_click)
        
        return tree

    def on_tree_double_click(self, event):
        """Handle double-click to populate form for editing."""
        tree = event.widget
        selection = tree.selection()
        if not selection:
            return
        
        # Get values from the row
        item = tree.item(selection[0])
        values = item['values']
        # values order: Qty(0), Name(1), Color(2), Category(3), Cost(4), PT(5), Sub(6), Key(7), Ab(8), Loy(9)
        
        name = values[1]
        category = values[3]
        
        # Fetch the FULL accurate data (incl. ID) from DB
        cursor = self.conn.cursor()
        cursor.execute("SELECT * FROM cards WHERE name=? AND card_category=?", (name, category))
        row = cursor.fetchone()
        
        if row:
            # DB Row: id(0), name(1), color(2), qty(3), cat(4), cost(5), pt(6), sub(7), key(8), ab(9), loy(10)
            self.editing_card_id = row[0] # Capture ID
            
            # Populate UI
            self.var_name.set(row[1])
            self.var_color.set(row[2] if row[2] else "")
            self.var_quantity.set(row[3])
            self.var_type.set(row[4])
            self.var_mana_cost.set(row[5] if row[5] else "")
            self.var_pt.set(row[6] if row[6] else "")
            self.var_subtypes.set(row[7] if row[7] else "")
            self.var_keywords.set(row[8] if row[8] else "")
            self.var_ability.set(row[9] if row[9] else "")
            self.var_loyalty.set(row[10] if row[10] else "")
            
            # Update UI State
            self.update_fields_visibility()
            self.btn_submit.config(text="Save Changes", bg="#b3d9ff", command=self.save_edits)
            # Enable deletion button visually (though logic handles check)
            self.btn_delete.config(state="normal")

    def save_edits(self):
        """Updates the existing record instead of creating a new one."""
        if self.editing_card_id is None:
            return
        
        # Gather data
        try:
            name = self.var_name.get().strip()
            cat = self.var_type.get()
            
            if not name or not cat:
                messagebox.showerror("Error", "Name and Category are required.")
                return

            cursor = self.conn.cursor()
            query = """
                UPDATE cards SET 
                name=?, color=?, quantity=?, card_category=?, mana_cost=?, 
                power_toughness=?, subtypes=?, keywords=?, ability=?, loyalty=?
                WHERE id=?
            """
            data = (
                name, self.var_color.get(), self.var_quantity.get(), cat,
                self.var_mana_cost.get(), self.var_pt.get(), self.var_subtypes.get(),
                self.var_keywords.get(), self.var_ability.get(), 
                self.var_loyalty.get() if self.var_loyalty.get() else None,
                self.editing_card_id
            )
            
            cursor.execute(query, data)
            self.conn.commit()
            
            messagebox.showinfo("Success", "Card updated successfully.")
            self.load_all_cards()
            self.clear_form() # Exits edit mode
            
        except sqlite3.IntegrityError:
            messagebox.showerror("Error", "A card with this Name and Category already exists. Cannot rename.")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def create_master_tab_content(self, parent):
        search_frame = tk.LabelFrame(parent, text="Filter Master List", padx=5, pady=5)
        search_frame.pack(fill="x", padx=5, pady=5)

        tk.Label(search_frame, text="Color:").pack(side="left")
        tk.Entry(search_frame, textvariable=self.search_color, width=10).pack(side="left", padx=5)
        
        tk.Label(search_frame, text="Category:").pack(side="left")
        self.search_cat_combo = ttk.Combobox(search_frame, textvariable=self.search_category, values=["All"] + self.card_types, state="readonly", width=15)
        self.search_cat_combo.current(0)
        self.search_cat_combo.pack(side="left", padx=5)

        tk.Label(search_frame, text="Subtype:").pack(side="left")
        tk.Entry(search_frame, textvariable=self.search_subtype, width=10).pack(side="left", padx=5)

        tk.Label(search_frame, text="Cost:").pack(side="left")
        tk.Entry(search_frame, textvariable=self.search_cost, width=8).pack(side="left", padx=5)

        tk.Button(search_frame, text="Search", command=self.search_master, bg="#dddddd").pack(side="left", padx=10)
        tk.Button(search_frame, text="Reset", command=self.clear_search, bg="#dddddd").pack(side="left")

        tree_frame = tk.Frame(parent)
        tree_frame.pack(fill="both", expand=True, padx=5, pady=5)
        
        if "Master" not in self.tree_dict: self.tree_dict["Master"] = {}
        self.tree_dict["Master"]["All"] = self.create_treeview(tree_frame)

    def create_color_tab_content(self, parent, color_name):
        inner_notebook = ttk.Notebook(parent)
        inner_notebook.pack(fill="both", expand=True, padx=5, pady=5)
        
        if color_name not in self.tree_dict:
            self.tree_dict[color_name] = {}

        all_frame = tk.Frame(inner_notebook)
        inner_notebook.add(all_frame, text=f" All {color_name} ")
        self.tree_dict[color_name]["All"] = self.create_treeview(all_frame)

        for ctype in self.card_types:
            type_frame = tk.Frame(inner_notebook)
            inner_notebook.add(type_frame, text=f" {ctype} ")
            self.tree_dict[color_name][ctype] = self.create_treeview(type_frame)

    def update_fields_visibility(self, event=None):
        for w in self.dynamic_frame.winfo_children():
            w.grid_forget()

        ctype = self.var_type.get()
        r = 0; col = 0
        show_mana = True; show_pt = False; show_sub = False
        show_key = False; show_ab = False; show_loy = False

        if ctype == "Land":
            show_mana = False; show_ab = True
        elif "Creature" in ctype or "Vehicle" in ctype: 
            show_pt = True; show_sub = True; show_key = True
        elif ctype == "Planeswalker":
            show_loy = True
        
        if ctype in ["Legendary Creature", "Creature", "Artifact Creature"]:
            show_sub = True; show_key = True

        widgets_map = [
            (show_mana, self.lbl_mana, self.ent_mana),
            (show_pt, self.lbl_pt, self.ent_pt),
            (show_sub, self.lbl_sub, self.ent_sub),
            (show_key, self.lbl_key, self.ent_key),
            (show_ab, self.lbl_ab, self.ent_ab),
            (show_loy, self.lbl_loy, self.ent_loy)
        ]

        for show, lbl, ent in widgets_map:
            if show:
                lbl.grid(row=r, column=col, sticky="e")
                ent.grid(row=r, column=col+1, sticky="w", padx=5)
                col += 2
                if col > 6:
                    r += 1; col = 0

    def clean_row(self, row):
        db_map = list(row)
        qty = db_map.pop(2)
        db_map.insert(0, qty)
        
        cleaned = []
        for item in db_map:
            if item is None: cleaned.append("")
            elif isinstance(item, str) and item.lower() == 'nan': cleaned.append("")
            else: cleaned.append(item)
        return cleaned

    def load_all_cards(self):
        for main_cat in self.tree_dict:
            for sub_cat in self.tree_dict[main_cat]:
                self.tree_dict[main_cat][sub_cat].delete(*self.tree_dict[main_cat][sub_cat].get_children())
        
        cursor = self.conn.cursor()
        cursor.execute("SELECT name, color, quantity, card_category, mana_cost, power_toughness, subtypes, keywords, ability, loyalty FROM cards ORDER BY name ASC")
        rows = cursor.fetchall()

        for row in rows:
            name = row[0]
            color_str = str(row[1]) if row[1] else "Colorless"
            category = row[3]
            display_values = self.clean_row(row)

            # 1. Master
            self.tree_dict["Master"]["All"].insert("", "end", values=display_values)

            # 2. Color Tabs
            dest_colors = []
            standard_colors = ["White", "Blue", "Black", "Red", "Green"]
            detected = [c for c in standard_colors if c in color_str]
            
            if len(detected) > 1:
                dest_colors.append("Multicolor")
                dest_colors.extend(detected)
            elif len(detected) == 1:
                dest_colors.append(detected[0])
            else:
                dest_colors.append("Colorless")

            for c_tab in dest_colors:
                if c_tab in self.tree_dict:
                    self.tree_dict[c_tab]["All"].insert("", "end", values=display_values)
                    if category in self.tree_dict[c_tab]:
                        self.tree_dict[c_tab][category].insert("", "end", values=display_values)

    def search_master(self):
        self.tree_dict["Master"]["All"].delete(*self.tree_dict["Master"]["All"].get_children())
        query = "SELECT name, color, quantity, card_category, mana_cost, power_toughness, subtypes, keywords, ability, loyalty FROM cards WHERE 1=1"
        params = []

        c_in = self.search_color.get().strip()
        if c_in:
            parts = c_in.replace('/', ' ').replace(',', ' ').split()
            for p in parts:
                query += " AND color LIKE ?"
                params.append(f"%{p}%")
        
        cat_in = self.search_category.get()
        if cat_in and cat_in != "All":
            query += " AND card_category = ?"
            params.append(cat_in)
            
        if self.search_subtype.get():
            query += " AND subtypes LIKE ?"
            params.append(f"%{self.search_subtype.get()}%")

        if self.search_cost.get():
            query += " AND mana_cost LIKE ?"
            params.append(f"%{self.search_cost.get()}%")

        query += " ORDER BY name ASC"
        cursor = self.conn.cursor()
        cursor.execute(query, params)
        for row in cursor.fetchall():
            self.tree_dict["Master"]["All"].insert("", "end", values=self.clean_row(row))

    def clear_search(self):
        self.search_color.set("")
        self.search_cost.set("")
        self.search_category.set("All")
        self.search_subtype.set("")
        self.load_all_cards()

    def clear_form(self):
        # Reset editing state
        self.editing_card_id = None
        self.btn_submit.config(text="Add Card", bg="#dddddd", command=self.add_card)
        
        self.var_name.set("")
        self.var_color.set("")
        self.var_quantity.set(1)
        self.var_type.set("")
        self.var_mana_cost.set("")
        self.var_pt.set("")
        self.var_subtypes.set("")
        self.var_keywords.set("")
        self.var_ability.set("")
        self.var_loyalty.set("")
        self.update_fields_visibility()

    def import_excel_dialog(self):
        file_path = filedialog.askopenfilename(filetypes=[("Excel Files", "*.xlsx *.xls")])
        if not file_path: return
        try:
            self.process_excel_file(file_path)
            self.conn.commit()
            self.load_all_cards()
            messagebox.showinfo("Import Complete", "Success!")
        except Exception as e:
            messagebox.showerror("Import Error", str(e))

    def process_excel_file(self, file_path):
        xls = pd.read_excel(file_path, sheet_name=None, header=None)
        
        cat_map = {
            "Not Basic Lands": "Land", "Legendary Creatures": "Legendary Creature",
            "Ledgendary Creatures": "Legendary Creature", "Creatures": "Creature",
            "Planeswalkers": "Planeswalker", "Instants": "Instant", "Sorceries": "Sorcery",
            "Legendary Enchantments - Backgrounds": "Legendary Enchantment",
            "Ledgendary Enchantments - Backgrounds": "Legendary Enchantment",
            "Legendary Enchantment Creature - Backgrounds": "Legendary Creature",
            "Ledgendary Enchantment Creature - Backgrounds": "Legendary Creature",
            "Enchantments": "Enchantment", "Legendary Artifact": "Legendary Artifact",
            "Ledgendary Artifact": "Legendary Artifact", "Ledgendary Artifacts": "Legendary Artifact",
            "Artifacts": "Artifact", "Artifact Equiment": "Artifact Equipment",
            "Artifact Vehicle": "Artifact Vehicle", "Artifact Treasure": "Artifact Treasure",
            "Atrifact Creatures": "Artifact Creature", "Articaft Cretures": "Artifact Creature"
        }

        for sheet_name, df in xls.items():
            if sheet_name in ["All", "Statistics"] or df.empty: continue
            
            num_cols = df.shape[1]
            j = 0
            while j < num_cols:
                cat_raw = df.iloc[0, j]
                if pd.notna(cat_raw) and isinstance(cat_raw, str) and cat_raw != "Total Cards":
                    mapped_cat = cat_map.get(cat_raw.strip(), cat_raw.strip())
                    
                    block_cols = []
                    k = j
                    while k < num_cols:
                        header = df.iloc[1, k]
                        if pd.isna(header): break
                        block_cols.append(str(header).strip())
                        k += 1
                    
                    if not block_cols: j += 1; continue

                    block_data = df.iloc[2:, j:k].copy()
                    block_data.columns = block_cols
                    if "Name" in block_data.columns:
                        block_data = block_data.dropna(subset=["Name"])
                        for _, row in block_data.iterrows():
                            c_val = row["Color"] if "Color" in row and pd.notna(row["Color"]) else sheet_name
                            data = {
                                "name": row["Name"], "card_category": mapped_cat,
                                "quantity": int(row["Quantity"]) if "Quantity" in row and pd.notna(row["Quantity"]) else 1,
                                "color": c_val,
                                "mana_cost": str(row["Cost"]) if "Cost" in row and pd.notna(row["Cost"]) else None,
                                "power_toughness": str(row["Power/Toughness"]) if "Power/Toughness" in row and pd.notna(row["Power/Toughness"]) else None,
                                "subtypes": str(row["Types"]) if "Types" in row and pd.notna(row["Types"]) else None,
                                "keywords": str(row["Keywords"]) if "Keywords" in row and pd.notna(row["Keywords"]) else None,
                                "ability": str(row["Ability"]) if "Ability" in row and pd.notna(row["Ability"]) else None,
                                "loyalty": int(row["Base Loyalty"]) if "Base Loyalty" in row and pd.notna(row["Base Loyalty"]) else None,
                            }
                            for key, val in data.items():
                                if str(val) == 'nan': data[key] = None
                            self.upsert_card(data)
                    j = k
                else: j += 1

: 

In [26]:
try:
    if 'root' in locals() and root:
        root.destroy()
except Exception:
    pass

root = tk.Tk()
app = MTGCardManager(root)
root.mainloop()